# MCP 複数Tool連携 — 整理版

このNotebookの目的は、**MCP Server / MCP Client / LLM / Tool の役割を混同せず説明できるようにすること**です。

元のNotebookにあった内容を、面接前に読み返しやすいように次の順番へ整理しています。

```text
1. 全体像
2. Server側
3. Client側
4. LLMがToolを選ぶ流れ
5. Tool実行
6. Tool結果から最終回答
7. 動作確認
8. 面接用の要点
```

## 1. まず全体像だけ覚える

```text
ユーザー質問
    ↓
OpenAI LLM
    ↑  利用可能なTool定義
MCP Client  ← list_tools() ←  MCP Server
    ↓
LLMがTool名・引数を選択
    ↓
MCP Client  → call_tool() → MCP Server → Toolを実行
    ↑                              ↓
    └──────── Tool結果 ────────────┘
    ↓
OpenAI LLMがTool結果を自然文へ整形
    ↓
最終回答
```

### 一番重要な役割分担

|役割|担当|
|---|---|
|Toolを公開する|MCP Server|
|Tool一覧を取得する|MCP Client (`list_tools()`)|
|どのToolを使うか判断する|LLM|
|選ばれたToolを呼び出す|MCP Client (`call_tool()`)|
|ファイル検索などの実処理をする|Server側のTool関数|
|Tool結果を自然文にする|LLM|

## 2. `server.py` — Toolを公開し、実処理を担当する

Serverは「Toolを置いておくだけの箱」ではありません。

**ClientへToolを公開し、呼び出されたToolを実際に実行する側**です。

Python関数に `@mcp.tool()` を付けると、その関数をMCP Toolとして公開できます。

In [ ]:
from pathlib import Path
from mcp.server import MCPServer

mcp = MCPServer("MCP File Tool Server")

@mcp.tool()
def search_folder_text(folder_path: str, keyword: str) -> dict[str, list[str]]:
    """指定したフォルダ直下のtxtファイルから、キーワードを含む行を検索します。"""
    results = {}

    for path in Path(folder_path).glob("*.txt"):
        lines = path.read_text(encoding="utf-8").splitlines()
        matched = [line for line in lines if keyword in line]

        if matched:
            results[path.name] = matched

    return results

### `@mcp.tool()` がしていること

```text
通常のPython関数
    ↓
@mcp.tool()
    ↓
MCP Clientから発見・呼び出し可能なToolとして登録
```

LLMがToolを選ぶ際には、主に次の情報が使われます。

```text
name         → Tool名
description  → docstringの説明
input_schema → 引数名・型などの入力仕様
```

そのため、**docstringはLLMがTool用途を判断する材料**でもあります。

## 3. `client.py` — ServerとLLMをつなぐ

Client側は主に5段階です。

```text
① list_tools() でServerのTool定義を取得
② OpenAI Function Calling形式へ変換
③ 質問 + Tool定義をLLMへ渡す
④ LLMが選んだToolを call_tool() で実行
⑤ Tool結果をLLMへ返して最終回答を生成
```

ここで重要なのは、**MCP Client自身が質問を読んでToolを選んでいるわけではない**ことです。

## 4. Step 1 — `list_tools()` はToolを実行しない

```python
tools_response = await mcp_client.list_tools()
```

`list_tools()` は、Serverへ次のように問い合わせる処理です。

> 「あなたが公開しているToolと、その定義を教えてください」

ここではToolの実処理はまだ行いません。

In [ ]:
from mcp import Client
from server import mcp

async with Client(mcp) as mcp_client:
    tools_response = await mcp_client.list_tools()

for tool in tools_response.tools:
    print("name       :", tool.name)
    print("description:", tool.description)
    print("input_schema:", tool.input_schema)
    print("---")

## 5. Step 2 — MCP Tool定義をOpenAI形式へ変換

MCPから取得したTool定義と、OpenAI Function Callingが受け取る形式は同じではありません。

そこで、`name`、`description`、`input_schema` をOpenAI側の形式へ組み替えます。

In [ ]:
def convert_mcp_tools_to_openai(mcp_tools):
    return [
        {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description or "",
                "parameters": tool.input_schema,
            },
        }
        for tool in mcp_tools
    ]

openai_tools = convert_mcp_tools_to_openai(tools_response.tools)
openai_tools

## 6. Step 3 — どのToolを使うかはLLMが判断する

質問と `openai_tools` をLLMへ渡します。

```python
response = openai_client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": question}],
    tools=openai_tools,
)
```

LLMは質問とTool定義を見て、必要ならTool呼び出しを返します。

**MCPがToolを選ぶのではありません。**

## 7. Step 4 — `tool_calls` からTool名と引数を取り出す

LLMがTool利用を選んだ場合、`tool_calls` に構造化された呼び出し情報が入ります。

```python
tool_call = response.choices[0].message.tool_calls[0]

tool_name = tool_call.function.name
tool_args = json.loads(tool_call.function.arguments)
```

```text
function.name
→ LLMが選んだTool名

function.arguments
→ そのToolへ渡す引数
```

これは単なる最終回答テキストではなく、**Toolを実行するための指示情報**です。

## 8. Step 5 — `call_tool()` で初めてToolを実行する

```python
tool_result = await mcp_client.call_tool(
    tool_name,
    tool_args,
)
```

ここで初めて、LLMが選んだToolを実行します。

例えば `search_folder_text` が選ばれた場合、実際にファイルを開いて検索するのは、**`server.py` に定義した `search_folder_text()`** です。

```text
Client
  call_tool("search_folder_text", args)
        ↓
MCP Server
        ↓
server.py の search_folder_text()
        ↓
ファイルを読み込んで検索
```

## 9. Step 6 — Tool結果をLLMへ戻す

Toolの生の結果をそのままユーザーへ返すのではなく、元の質問と一緒にLLMへ戻します。

目的は単なるJSON変換ではなく、**Tool結果を質問の文脈に合わせた自然な最終回答へ整えること**です。

```text
元の質問
+ LLMが行ったTool call
+ Toolの実行結果
        ↓
LLM
        ↓
ユーザー向けの最終回答
```

## 10. `list_tools()` と `call_tool()` を混同しない

```text
list_tools()
→ ServerにどんなToolがあるか取得する
→ 実行前

call_tool()
→ LLMが選んだToolを実際に実行する
→ 実行時
```

短く覚えるなら、

> **list = 発見、call = 実行**

です。

## 11. 完成版 `client.py` の処理順

完成版では処理を関数にまとめていますが、順番は同じです。

```text
ask_mcp(question)
│
├─ 1. list_tools()
├─ 2. OpenAI形式へ変換
├─ 3. LLMへ質問とToolを渡す
├─ 4. tool_callsから名前・引数を取得
├─ 5. call_tool()
└─ 6. Tool結果をLLMへ返して最終回答
```

コード全体は同梱の `client.py` を参照してください。

In [ ]:
# Notebookから完成版を呼ぶ場合の例
from client import ask_mcp, DATA_DIR

question = f"{DATA_DIR} の中から、MCPという文字を含む行を探してください。"
answer = await ask_mcp(question)
print(answer)

## 12. 動作確認用の質問例

Toolごとの分岐をClientへ書かず、LLMがTool定義から選べることを確認します。

```python
# list_files を期待
f"{DATA_DIR} にあるファイル一覧を教えてください。"

# read_text_file を期待
f"{DATA_DIR / 'AI利用ルール.txt'} の内容を読んでください。"

# get_file_info を期待
f"{DATA_DIR / 'PC申請.txt'} のファイル情報を教えてください。"

# search_folder_text を期待
f"{DATA_DIR} の中から、MCPという文字を含む行を探してください。"

# count_keyword を期待
f"{DATA_DIR} の各ファイルで、申請という文字が何回出るか数えてください。"
```

LLMの判断は確率的なので、期待したToolが必ず選択されるとは限りません。Tool名やdocstringを分かりやすくすることが重要です。

## 13. 面接で最低限説明できればよいポイント

### MCP Serverとは
Toolを保存するだけではなく、**Toolを公開し、呼び出された処理を実行する側**。

### MCP Clientとは
ServerからTool定義を取得し、選ばれたToolの実行をServerへ依頼する側。

### Toolを選ぶのは誰か
今回の構成では**OpenAIのLLM**。

### `list_tools()` と `call_tool()`
`list_tools()` はToolの発見、`call_tool()` はToolの実行。

### Function Calling形式への変換が必要な理由
MCP Tool定義を、OpenAIが利用可能なToolとして認識できる形式に合わせるため。

### Tool結果を再度LLMへ渡す理由
Toolの結果を、元の質問に沿ったユーザー向けの自然な回答へ整えるため。

### ServerへToolを追加した場合
`@mcp.tool()` で公開すれば `list_tools()` から取得できるため、ClientへToolごとの固定的な `if / elif` を追加せず拡張しやすい。

## 14. 今回あえて簡略化している点

このポートフォリオはMCPの基本フローを説明することを優先しています。

- `tool_calls[0]` の1件だけを処理
- `tool_result.content[0].text` を利用
- 詳細な例外処理は省略
- `Client(mcp)` による同一プロセス接続

実運用では、複数Tool呼び出し、Toolエラー、権限制御、stdio / Streamable HTTPなどを追加検討できます。

まずは、

```text
ServerでTool公開
→ ClientがTool発見
→ LLMがTool選択
→ ClientがTool実行を依頼
→ Server側Toolが実処理
→ LLMが最終回答
```

を説明できれば、今回の実装の中心は押さえられています。